In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Works for both 3D tensors (batch, seq_len, d_k)
    and 4D tensors (batch, num_heads, seq_len, d_k).
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, V)
    return output, attention_weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        batch_size, seq_len, _ = x.size()
        x = x.view(batch_size, seq_len, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_len, _ = x.size()
        x = x.transpose(1, 2).contiguous()
        return x.view(batch_size, seq_len, self.d_model)

    def forward(self, query, key, value, mask=None):
        Q = self.split_heads(self.W_q(query))
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))

        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        concat_output = self.combine_heads(attn_output)
        output = self.W_o(concat_output)
        return output, attn_weights

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Dimension per head

        # Linear projections for Query, Key, Value, and final Output
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        """
        Reshapes (batch_size, seq_len, d_model) -> (batch_size, num_heads, seq_len, d_k)
        so all heads can be computed in parallel via batched matrix multiplication.
        """
        batch_size, seq_len, _ = x.size()
        x = x.view(batch_size, seq_len, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def combine_heads(self, x):
        """
        Reverses split_heads:
        (batch_size, num_heads, seq_len, d_k) -> (batch_size, seq_len, d_model)
        """
        batch_size, _, seq_len, _ = x.size()
        x = x.transpose(1, 2).contiguous()
        return x.view(batch_size, seq_len, self.d_model)

    def forward(self, query, key, value, mask=None):
        # 1. Project inputs into Q, K, V spaces: (batch, seq_len, d_model)
        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        # 2. Split into multiple heads: (batch, num_heads, seq_len, d_k)
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)

        # 3. Apply Scaled Dot-Product Attention across all heads in parallel
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)

        # 4. Concatenate heads back together: (batch, seq_len, d_model)
        concat_output = self.combine_heads(attn_output)

        # 5. Final linear mixing layer
        output = self.W_o(concat_output)

        return output, attn_weights

In [3]:
torch.manual_seed(42)

# Hyperparameters
batch_size = 1
seq_len = 3      # e.g., 4 tokens in a sentence
d_model = 4     # Embedding size per token
num_heads = 4    # 4 heads -> d_k = 16 / 4 = 4 dimensions per head

# Simulated input embeddings: (batch_size, seq_len, d_model)
x = torch.randn(batch_size, seq_len, d_model)

# Initialize Multi-Head Attention
mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)

# --- Test 1: Unmasked Self-Attention (Encoder-style) ---
output, attn_weights = mha(query=x, key=x, value=x)
print("Input shape:            ", x.shape)
print("Output shape:           ", output.shape)

print("Attention weights shape:", attn_weights.shape)
print("Attention Weights:")
print(attn_weights, "\n")

print("Attention Output:")
print(output, "\n")
# --- Test 2: Causal Masking (Decoder / GPT-style) ---
# Lower-triangular matrix ensures token i can only attend to tokens <= i
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)
masked_output, masked_weights = mha(query=x, key=x, value=x, mask=causal_mask)

print(f"Input Shape: {x.shape}")
print(f"Number of Heads: {num_heads}")
print(f"Attention Weights Shape: {masked_weights.shape}\n")

print("Multi-Head Attention Output:")
print(masked_output, "\n")

print(f"Output Shape: {masked_output.shape}")

Input shape:             torch.Size([1, 3, 4])
Output shape:            torch.Size([1, 3, 4])
Attention weights shape: torch.Size([1, 4, 3, 3])
Attention Weights:
tensor([[[[0.3375, 0.3314, 0.3312],
          [0.3178, 0.3407, 0.3416],
          [0.3470, 0.3269, 0.3262]],

         [[0.3070, 0.3712, 0.3219],
          [0.4978, 0.1398, 0.3624],
          [0.2811, 0.4099, 0.3089]],

         [[0.3034, 0.3726, 0.3241],
          [0.3959, 0.2589, 0.3453],
          [0.2887, 0.3925, 0.3187]],

         [[0.2858, 0.4382, 0.2760],
          [0.2854, 0.4390, 0.2756],
          [0.2867, 0.4362, 0.2771]]]], grad_fn=<SoftmaxBackward0>) 

Attention Output:
tensor([[[-0.4398,  0.0742,  0.1407, -0.3353],
         [-0.3663,  0.0420, -0.0021, -0.3062],
         [-0.4527,  0.0801,  0.1638, -0.3408]]], grad_fn=<ViewBackward0>) 

Input Shape: torch.Size([1, 3, 4])
Number of Heads: 4
Attention Weights Shape: torch.Size([1, 4, 3, 3])

Multi-Head Attention Output:
tensor([[[-0.1976,  0.1590, -0.2575, -0.3502